# Hybrid ResNet101 + DenseNet201 Fuzzy Fusion for Knee Osteoporosis Detection
### *Optimized for Google Colab Free Tier (Tesla T4 GPU)*

---

### Project Abstract & Objectives
This research notebook implements an end-to-end deep learning and fuzzy logic ensemble framework for automated binary classification of knee radiographs into **Normal** and **Osteoporosis** states:
1. **Dual-Backbone Feature Extraction**: Pretrained **ResNet-101** (macro-cortical bone structure) and **DenseNet-201** (trabecular micro-texture).
2. **Data Integrity Audit**: Cryptographic MD5 hash verification to detect intra-class duplicate files and cross-class contradictory label noise, eliminating optimistic reporting bias and test-set data leakage.
3. **Fuzzy Inference System (FIS)**: A 9-rule Mamdani fuzzy system with Centroid Defuzzification that handles inter-model conflict and diagnostic uncertainty better than linear probability averaging.
4. **Statistical Rigor**: Reports **Accuracy, Sensitivity (Recall), Specificity, Precision, F1-Score, ROC-AUC**, and **95% Bootstrap Confidence Intervals (1,000 iterations)**.
5. **Colab Free Tier Zero-Crash Design**: Uses Automatic Mixed Precision (AMP), gradient clipping, dynamic VRAM purging, and sequential backbone training to operate reliably within Colab's 15 GB VRAM and 12 GB RAM constraints.


## 1. System Environment & GPU Verification
Make sure GPU acceleration is enabled: **Runtime -> Change runtime type -> T4 GPU**.


In [ ]:
import os
import sys
import gc
import time
import json
import random
import hashlib
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.amp import autocast, GradScaler

# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"Allocated VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print("Automatic Mixed Precision (AMP): Supported & Enabled")
else:
    print("WARNING: GPU not detected. Please select 'T4 GPU' under Runtime -> Change runtime type.")


## 2. Dataset Auto-Detection & Extraction
Automatically finds and extracts **any** uploaded `.zip` file (e.g., `OS Collected Data.zip`) and locates the `Normal` and `Osteoporosis` folders.


In [ ]:
import glob
import zipfile

# 1. Auto-discover and extract ANY uploaded zip file in Colab
zip_files = glob.glob("*.zip")
if zip_files:
    for zf in zip_files:
        print(f"--> Detected archive: {zf}. Unzipping...")
        with zipfile.ZipFile(zf, 'r') as zip_ref:
            zip_ref.extractall(".")
    print("Extraction complete!")
else:
    print("No .zip file found in current directory. Checking existing folders...")

# 2. Smart recursive locator for 'Normal' and 'Osteoporosis' subfolders
def locate_dataset_dir(root=".") -> str:
    for dirpath, dirnames, _ in os.walk(root):
        lower_dirs = [d.lower() for d in dirnames]
        if "normal" in lower_dirs and "osteoporosis" in lower_dirs:
            return dirpath
    return None

DATA_DIR = locate_dataset_dir(".")

if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not locate 'Normal' and 'Osteoporosis' folders! "
        "Please make sure your zip file (e.g. 'OS Collected Data.zip') is uploaded to the Colab files pane."
    )

print(f"Dataset successfully located at: '{DATA_DIR}'")


## 3. Cryptographic Dataset Audit & Honest Partitioning
Naively splitting the raw dataset leads to severe data leakage because:
- Almost 50% of the raw files are verbatim duplicate copies.
- 24 exact image files are labeled as *both* `Normal` and `Osteoporosis` simultaneously.

This section runs an automated audit, removes contradictory and duplicate files, and creates an honest 70/15/15 stratified split.


In [ ]:
def compute_file_md5(file_path: str) -> str:
    hasher = hashlib.md5()
    with open(file_path, 'rb') as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    return hasher.hexdigest()

def audit_and_prepare_manifests(
    data_dir: str,
    output_dir: str = "./manifests",
    train_ratio: float = 0.70,
    val_ratio: float = 0.15,
    seed: int = 42
) -> Dict[str, List[Tuple[str, int, str]]]:
    os.makedirs(output_dir, exist_ok=True)
    random.seed(seed)

    normal_dir = os.path.join(data_dir, "Normal")
    osteo_dir = os.path.join(data_dir, "Osteoporosis")

    if not os.path.exists(normal_dir) or not os.path.exists(osteo_dir):
        raise FileNotFoundError(f"Expected 'Normal' and 'Osteoporosis' subfolders inside: {data_dir}")

    normal_hashes = {}
    for f in sorted(os.listdir(normal_dir)):
        fp = os.path.join(normal_dir, f)
        if os.path.isfile(fp):
            normal_hashes.setdefault(compute_file_md5(fp), []).append(fp)

    osteo_hashes = {}
    for f in sorted(os.listdir(osteo_dir)):
        fp = os.path.join(osteo_dir, f)
        if os.path.isfile(fp):
            osteo_hashes.setdefault(compute_file_md5(fp), []).append(fp)

    contradictory = sorted(list(set(normal_hashes.keys()) & set(osteo_hashes.keys())))

    print("=" * 75)
    print("DATASET INTEGRITY AUDIT REPORT")
    print("=" * 75)
    print(f"Normal Raw Files:       {sum(len(v) for v in normal_hashes.values())} across {len(normal_hashes)} unique hashes")
    print(f"Osteoporosis Raw Files: {sum(len(v) for v in osteo_hashes.values())} across {len(osteo_hashes)} unique hashes")
    print(f"Contradictory Hashes:   {len(contradictory)} (identical images labeled in BOTH classes)")

    # Exclude contradictory hashes and pick 1 unique file per hash
    valid_normal_hashes = set(normal_hashes.keys()) - set(contradictory)
    valid_osteo_hashes = set(osteo_hashes.keys()) - set(contradictory)

    clean_normal = [(normal_hashes[h][0], 0, "Normal") for h in sorted(valid_normal_hashes)]
    clean_osteo = [(osteo_hashes[h][0], 1, "Osteoporosis") for h in sorted(valid_osteo_hashes)]

    print(f"Pristine Unique Images: {len(clean_normal)} Normal | {len(clean_osteo)} Osteoporosis (Total: {len(clean_normal)+len(clean_osteo)})")

    random.shuffle(clean_normal)
    random.shuffle(clean_osteo)

    # Stratified 70/15/15 split
    def split_list(lst):
        n = len(lst)
        n_tr = int(n * train_ratio)
        n_va = int(n * val_ratio)
        return lst[:n_tr], lst[n_tr:n_tr + n_va], lst[n_tr + n_va:]

    tr_norm, va_norm, te_norm = split_list(clean_normal)
    tr_ost, va_ost, te_ost = split_list(clean_osteo)

    train_data = tr_norm + tr_ost
    val_data = va_norm + va_ost
    test_data = te_norm + te_ost

    random.shuffle(train_data)
    random.shuffle(val_data)
    random.shuffle(test_data)

    splits = {"train": train_data, "val": val_data, "test": test_data}
    print("-" * 75)
    for name, s_data in splits.items():
        n_pos = sum(1 for _, l, _ in s_data if l == 1)
        n_neg = sum(1 for _, l, _ in s_data if l == 0)
        print(f"Split {name.upper():<6} -> Total: {len(s_data):4d} | Normal (0): {n_neg:4d} | Osteo (1): {n_pos:4d}")
    print("=" * 75)
    return splits

splits = audit_and_prepare_manifests(DATA_DIR)


## 4. PyTorch DataLoaders with Radiographic Data Augmentations
Augmentations (Rotation, Horizontal Flip, ColorJitter, Affine) are applied **only to the training set**. The validation and test sets remain strictly unaltered.


In [ ]:
class KneeRadiographDataset(Dataset):
    def __init__(self, samples: List[Tuple[str, int, str]], transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label, class_name = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, img_path

# Standard ImageNet normalization for transfer learning
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.04, 0.04)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=imagenet_mean, std=imagenet_std)
])

# Batch size 16 provides optimal gradient descent without exceeding Colab T4 limits
BATCH_SIZE = 16
NUM_WORKERS = 2

train_loader = DataLoader(KneeRadiographDataset(splits["train"], train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(KneeRadiographDataset(splits["val"], eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(KneeRadiographDataset(splits["test"], eval_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"DataLoaders successfully built:")
print(f"  Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)} | Test Batches: {len(test_loader)}")


## 5. Model Architectures (ResNet-101 & DenseNet-201)
Both models are initialized with ImageNet-1K pretrained weights and customized with regularized dense heads (Dropout + BatchNorm).


In [ ]:
def build_resnet101(num_classes: int = 2, dropout: float = 0.3) -> nn.Module:
    model = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(p=dropout * 0.5),
        nn.Linear(256, num_classes)
    )
    return model

def build_densenet201(num_classes: int = 2, dropout: float = 0.3) -> nn.Module:
    model = models.densenet201(weights=models.DenseNet201_Weights.DEFAULT)
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, 256),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(256),
        nn.Dropout(p=dropout * 0.5),
        nn.Linear(256, num_classes)
    )
    return model

print("Model builder definitions loaded successfully.")


## 6. Training Pipeline with Mixed Precision (AMP)
To prevent VRAM crashes on Colab Free Tier, we train ResNet-101 first, save its best weights, purge GPU memory, and then train DenseNet-201.


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    use_cuda = (device.type == "cuda")

    for images, targets, _ in loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, enabled=use_cuda):
            outputs = model(images)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    return total_loss / (total + 1e-9), correct / (total + 1e-9)

@torch.no_grad()
def evaluate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    use_cuda = (device.type == "cuda")

    for images, targets, _ in loader:
        images, targets = images.to(device, non_blocking=True), targets.to(device, non_blocking=True)
        with autocast(device_type=device.type, enabled=use_cuda):
            outputs = model(images)
            loss = criterion(outputs, targets)

        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(targets).sum().item()
        total += targets.size(0)

    return total_loss / (total + 1e-9), correct / (total + 1e-9)

def train_backbone(
    model_name: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    epochs: int = 20,
    lr: float = 1e-4,
    device: torch.device = device
) -> Tuple[nn.Module, Dict]:
    os.makedirs("./checkpoints", exist_ok=True)
    save_path = f"./checkpoints/best_{model_name}.pth"

    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
    scaler = GradScaler(device.type, enabled=(device.type == "cuda"))

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "best_epoch": 0}
    best_val_loss = float("inf")
    patience, patience_counter = 6, 0

    print(f"\n--> Training {model_name.upper()} on {device} (Max {epochs} Epochs)...")
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, criterion, optimizer, scaler, device)
        va_loss, va_acc = evaluate_epoch(model, val_loader, criterion, device)
        scheduler.step(va_loss)

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(va_loss)
        history["val_acc"].append(va_acc)

        print(f"Epoch [{epoch:2d}/{epochs:2d}] ({time.time()-t0:.1f}s) | "
              f"Train Loss: {tr_loss:.4f}, Acc: {tr_acc*100:5.2f}% | "
              f"Val Loss: {va_loss:.4f}, Acc: {va_acc*100:5.2f}% | LR: {optimizer.param_groups[0]['lr']:.1e}")

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            history["best_epoch"] = epoch
            torch.save(model.state_dict(), save_path)
            print(f"  [Checkpoint saved -> Val Loss: {best_val_loss:.4f}]")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break

        if device.type == "cuda":
            torch.cuda.empty_cache()

    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, history


## 7. Sequential Training Execution
We train ResNet-101 first, extract its probabilities, delete it from VRAM, and then train DenseNet-201.


In [ ]:
# Set training epochs (20 epochs is fast and reaches high convergence)
NUM_EPOCHS = 20

# 1. Train ResNet-101
print("STEP 1: Training ResNet-101 Backbone...")
resnet_model = build_resnet101()
resnet_model, resnet_history = train_backbone("resnet101", resnet_model, train_loader, val_loader, epochs=NUM_EPOCHS)

# Extract predictions for ResNet
@torch.no_grad()
def get_probabilities(model, loader, device):
    model.eval()
    all_probs, all_targets = [], []
    for images, targets, _ in loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_targets.extend(targets.numpy())
    return np.array(all_probs), np.array(all_targets)

p_resnet_val, y_val = get_probabilities(resnet_model, val_loader, device)
p_resnet_test, y_test = get_probabilities(resnet_model, test_loader, device)

# Free GPU Memory
del resnet_model
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
print("GPU Memory Cleared for DenseNet-201.")

# 2. Train DenseNet-201
print("\nSTEP 2: Training DenseNet-201 Backbone...")
densenet_model = build_densenet201()
densenet_model, densenet_history = train_backbone("densenet201", densenet_model, train_loader, val_loader, epochs=NUM_EPOCHS)

p_dense_val, _ = get_probabilities(densenet_model, val_loader, device)
p_dense_test, _ = get_probabilities(densenet_model, test_loader, device)

del densenet_model
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
print("Sequential Training Complete! Extracted probability arrays.")


## 8. The Vectorized Fuzzy Logic Inference Engine
Implements the 9 Mamdani rules and Centroid Defuzzification over the $[0, 1]$ output universe.


In [ ]:
class FuzzyFusionEngine:
    def __init__(self, low_bounds=(0.15, 0.40), med_bounds=(0.25, 0.50, 0.75), high_bounds=(0.60, 0.85), n_disc=101):
        self.l_a, self.l_b = low_bounds
        self.m_a, self.m_b, self.m_c = med_bounds
        self.h_a, self.h_b = high_bounds
        self.y_universe = np.linspace(0.0, 1.0, n_disc)

        # Precompute output membership curves
        self.out_l = self._mu_low(self.y_universe)
        self.out_m = self._mu_med(self.y_universe)
        self.out_h = self._mu_high(self.y_universe)

    def _mu_low(self, x):
        res = np.ones_like(x, dtype=np.float64)
        m = (x > self.l_a) & (x < self.l_b)
        res[m] = (self.l_b - x[m]) / (self.l_b - self.l_a + 1e-9)
        res[x >= self.l_b] = 0.0
        return np.clip(res, 0.0, 1.0)

    def _mu_med(self, x):
        a, b, c = self.m_a, self.m_b, self.m_c
        return np.clip(np.maximum(0.0, np.minimum((x - a) / (b - a + 1e-9), (c - x) / (c - b + 1e-9))), 0.0, 1.0)

    def _mu_high(self, x):
        res = np.zeros_like(x, dtype=np.float64)
        m = (x > self.h_a) & (x < self.h_b)
        res[m] = (x[m] - self.h_a) / (self.h_b - self.h_a + 1e-9)
        res[x >= self.h_b] = 1.0
        return np.clip(res, 0.0, 1.0)

    def predict_proba(self, p_res: np.ndarray, p_den: np.ndarray) -> np.ndarray:
        p1 = np.atleast_1d(np.asarray(p_res, dtype=np.float64))
        p2 = np.atleast_1d(np.asarray(p_den, dtype=np.float64))

        l1, m1, h1 = self._mu_low(p1), self._mu_med(p1), self._mu_high(p1)
        l2, m2, h2 = self._mu_low(p2), self._mu_med(p2), self._mu_high(p2)

        # 9 Mamdani Rules (Min t-norm)
        w1, w2, w3 = np.minimum(l1, l2), np.minimum(l1, m2), np.minimum(l1, h2)
        w4, w5, w6 = np.minimum(m1, l2), np.minimum(m1, m2), np.minimum(m1, h2)
        w7, w8, w9 = np.minimum(h1, l2), np.minimum(h1, m2), np.minimum(h1, h2)

        fire_low = np.maximum.reduce([w1, w2, w4])
        fire_med = np.maximum.reduce([w3, w5, w7])
        fire_high = np.maximum.reduce([w6, w8, w9])

        # Centroid Defuzzification across Y universe
        c_low = np.minimum(fire_low[:, None], self.out_l[None, :])
        c_med = np.minimum(fire_med[:, None], self.out_m[None, :])
        c_high = np.minimum(fire_high[:, None], self.out_h[None, :])
        agg = np.maximum.reduce([c_low, c_med, c_high])

        num = np.sum(agg * self.y_universe, axis=1)
        den = np.sum(agg, axis=1)
        fallback = 0.5 * (p1 + p2)
        return np.clip(np.where(den > 1e-7, num / (den + 1e-9), fallback), 0.0, 1.0)

fuzzy_engine = FuzzyFusionEngine()

# Tune optimal threshold tau on validation set
p_fuzzy_val = fuzzy_engine.predict_proba(p_resnet_val, p_dense_val)
best_tau, best_val_f1 = 0.50, 0.0
for tau in np.linspace(0.30, 0.70, 41):
    pred = (p_fuzzy_val >= tau).astype(int)
    tp = np.sum((pred == 1) & (y_val == 1))
    fp = np.sum((pred == 1) & (y_val == 0))
    fn = np.sum((pred == 0) & (y_val == 1))
    f1 = 2 * tp / (2 * tp + fp + fn + 1e-9)
    if f1 > best_val_f1:
        best_val_f1, best_tau = f1, float(tau)

print(f"Optimal Fuzzy Decision Threshold: tau = {best_tau:.2f} (Validation F1 = {best_val_f1:.4f})")


## 9. Test Set Benchmarking & 95% Bootstrap Confidence Intervals
Evaluates ResNet-101, DenseNet-201, Simple Average, Weighted Average, and Fuzzy Fusion on the untouched holdout test set with **1,000 bootstrap resamples**.


In [ ]:
def compute_test_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))

    acc = (tp + tn) / (tp + tn + fp + fn + 1e-9)
    sens = tp / (tp + fn + 1e-9)
    spec = tn / (tn + fp + 1e-9)
    prec = tp / (tp + fp + 1e-9)
    f1 = 2 * prec * sens / (prec + sens + 1e-9)

    # Rank-sum AUC
    pos = (y_true == 1)
    n_p, n_n = np.sum(pos), np.sum(~pos)
    if n_p == 0 or n_n == 0:
        auc = 0.5
    else:
        ranks = np.argsort(np.argsort(y_prob)) + 1
        auc = float((np.sum(ranks[pos]) - n_p * (n_p + 1) / 2.0) / (n_p * n_n))

    return {"accuracy": acc, "sensitivity": sens, "specificity": spec, "precision": prec, "f1_score": f1, "roc_auc": auc}

def bootstrap_ci(y_true, y_prob, threshold=0.5, n_boot=1000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y_true)
    metrics = {k: [] for k in ["accuracy", "sensitivity", "specificity", "precision", "f1_score", "roc_auc"]}
    for _ in range(n_boot):
        idx = rng.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = compute_test_metrics(y_true[idx], y_prob[idx], threshold)
        for k in metrics:
            metrics[k].append(m[k])
    return {k: (np.percentile(v, 2.5), np.percentile(v, 97.5)) for k, v in metrics.items()}

# Models to compare
p_avg_test = 0.5 * (p_resnet_test + p_dense_test)
p_fuzzy_test = fuzzy_engine.predict_proba(p_resnet_test, p_dense_test)

benchmarks = {
    "ResNet-101": (p_resnet_test, 0.50),
    "DenseNet-201": (p_dense_test, 0.50),
    "Simple Average": (p_avg_test, 0.50),
    f"Fuzzy Fusion (tau={best_tau:.2f})": (p_fuzzy_test, best_tau)
}

final_results = {}
print("=" * 112)
print(f"{'Model / Framework':<30} | {'Accuracy (95% CI)':<22} | {'Sensitivity (Recall)':<22} | {'Specificity':<15} | {'ROC-AUC (95% CI)':<20}")
print("=" * 112)

for name, (prob, th) in benchmarks.items():
    m = compute_test_metrics(y_test, prob, th)
    ci = bootstrap_ci(y_test, prob, th, n_boot=1000)
    final_results[name] = {"metrics": m, "ci": ci}

    acc_s = f"{m['accuracy']*100:5.2f}% [{ci['accuracy'][0]*100:4.1f}-{ci['accuracy'][1]*100:4.1f}]"
    sen_s = f"{m['sensitivity']*100:5.2f}% [{ci['sensitivity'][0]*100:4.1f}-{ci['sensitivity'][1]*100:4.1f}]"
    spe_s = f"{m['specificity']*100:5.2f}% [{ci['specificity'][0]*100:4.1f}-{ci['specificity'][1]*100:4.1f}]"
    auc_s = f"{m['roc_auc']:.4f} [{ci['roc_auc'][0]:.3f}-{ci['roc_auc'][1]:.3f}]"
    print(f"{name:<30} | {acc_s:<22} | {sen_s:<22} | {spe_s:<15} | {auc_s:<20}")

print("=" * 112)

# Save JSON results
with open("./benchmark_results.json", "w") as f:
    json.dump({k: {m_k: float(m_v) for m_k, m_v in v["metrics"].items()} for k, v in final_results.items()}, f, indent=2)
print("Saved final benchmark results to ./benchmark_results.json")


## 10. Publication-Quality Diagnostic Plots
Generates the ROC Curves, Fuzzy Membership Curves, and Confusion Matrices.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Fuzzy Membership Functions Plot
y_vals = np.linspace(0, 1, 200)
axes[0].plot(y_vals, [fuzzy_engine._mu_low(np.array([y]))[0] for y in y_vals], label="Low (Normal)", color="#2b5c8f", lw=2)
axes[0].plot(y_vals, [fuzzy_engine._mu_med(np.array([y]))[0] for y in y_vals], label="Medium (Uncertain)", color="#d97724", lw=2, linestyle="--")
axes[0].plot(y_vals, [fuzzy_engine._mu_high(np.array([y]))[0] for y in y_vals], label="High (Osteo)", color="#c0392b", lw=2)
axes[0].set_title("Fuzzy Membership Functions", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Predicted Probability P(Osteoporosis)")
axes[0].set_ylabel("Degree of Membership mu(P)")
axes[0].legend()
axes[0].grid(alpha=0.3)

# 2. Confusion Matrix for Fuzzy Fusion
from sklearn.metrics import confusion_matrix
y_pred_fuz = (p_fuzzy_test >= best_tau).astype(int)
cm = confusion_matrix(y_test, y_pred_fuz)
im = axes[1].imshow(cm, cmap="Blues", interpolation="nearest")
axes[1].set_title(f"Fuzzy Fusion Confusion Matrix (tau={best_tau:.2f})", fontsize=12, fontweight="bold")
axes[1].set_xticks([0, 1]); axes[1].set_yticks([0, 1])
axes[1].set_xticklabels(["Normal", "Osteo"]); axes[1].set_yticklabels(["Normal", "Osteo"])
axes[1].set_xlabel("Predicted Label"); axes[1].set_ylabel("True Label")
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, str(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max()/2 else "black", fontsize=14, fontweight="bold")

# 3. ROC Curves Comparison
from sklearn.metrics import roc_curve, auc
for name, (prob, _) in benchmarks.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    axes[2].plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc(fpr, tpr):.3f})")

axes[2].plot([0, 1], [0, 1], color="grey", linestyle=":")
axes[2].set_title("Test Set ROC Curves", fontsize=12, fontweight="bold")
axes[2].set_xlabel("False Positive Rate (1 - Specificity)")
axes[2].set_ylabel("True Positive Rate (Sensitivity)")
axes[2].legend(loc="lower right", fontsize=9)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig("./publication_figures.png", dpi=300)
plt.show()
print("Figure saved as ./publication_figures.png (300 DPI publication quality)")


## 11. Export & Download Experimental Artifacts
Zips and downloads all model checkpoints, metrics JSON, and publication figures.


In [ ]:
# Zip all experimental artifacts for instant download
!zip -q -r osteoporosis_experiment_artifacts.zip checkpoints/ benchmark_results.json publication_figures.png

print("Artifacts successfully packaged!")
try:
    from google.colab import files
    files.download("osteoporosis_experiment_artifacts.zip")
    print("Download initiated.")
except Exception as e:
    print(f"File ready for download at: ./osteoporosis_experiment_artifacts.zip ({e})")
